# Treinamento de Modelos — Dataset de Obesidade

Este notebook demonstra o treinamento, comparação e avaliação de três modelos de classificação para predição de obesidade, usando otimização de hiperparâmetros via Optuna e rastreabilidade via MLflow.

**Objetivos:**
- Treinar Random Forest, XGBoost e MLP com Optuna (TPESampler, f1_macro)
- Comparar os modelos com tabela de métricas (accuracy, macro-F1, per-class F1)
- Selecionar o melhor modelo e retreinar em train+val completo
- Avaliar no test set com todas as métricas obrigatórias
- Plotar matriz de confusão 7×7, curvas ROC, feature importance, correlação de Spearman
- Logar experimento no MLflow: parâmetros, métricas, pipeline.joblib como artefato
- Serializar pipeline.joblib em models/

**Requisitos atendidos:** 7.1–7.6, 8.1–8.6, 9.1–9.5, 13.3, 13.4, 13.5

In [ ]:
# Importações
import os
import sys
import json
import logging
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import mlflow
import mlflow.sklearn
import optuna
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import classification_report

# Suprimir warnings desnecessários
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Configurações de visualização
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11
sns.set_theme(style='whitegrid', palette='muted')

# Logging básico para o notebook
logging.basicConfig(level=logging.INFO, format='%(levelname)s — %(message)s')
logger = logging.getLogger('model_training')

print('Importações concluídas.')

In [ ]:
# Adicionar raiz do projeto ao sys.path para importar src/
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data.validator import DataValidator
from src.data.cleaner import DataCleaner
from src.features.engineer import FeatureEngineer
from src.models.trainer import (
    train_with_optuna, build_pipeline, compare_models, ModelType
)
from src.models.evaluator import (
    evaluate, plot_confusion_matrix, plot_roc_curves,
    plot_feature_importance, plot_spearman_correlation
)
from src.config import (
    OBESITY_LABELS_PT, RANDOM_STATE, CATEGORICAL_VALUES
)

print(f'Raiz do projeto: {PROJECT_ROOT}')
print('Módulos src/ importados com sucesso.')

## Carregamento e Preparação dos Dados

Reproduzimos o mesmo pipeline de pré-processamento do notebook 02: carregamento do CSV, validação, limpeza e split estratificado 70/15/15 com `random_state=42`.

In [ ]:
# Carregar, validar e limpar os dados
DATA_PATH = os.path.join(PROJECT_ROOT, 'data', 'Obesity.csv')

df_raw = pd.read_csv(DATA_PATH)
logger.info('CSV carregado. Shape: %s', df_raw.shape)

validator = DataValidator()
validation_result = validator.validate(df_raw)

if validation_result.is_valid:
    print('✅ Validação aprovada — nenhum erro encontrado.')
else:
    print(f'⚠️  Validação com {len(validation_result.errors)} aviso(s):')
    for err in validation_result.errors[:10]:
        print(f'   • {err}')

cleaner = DataCleaner()
df = cleaner.clean(df_raw)

print(f'\nShape após limpeza: {df.shape}')
print(f'Duplicatas removidas: {len(df_raw) - len(df)}')
print(f'Valores ausentes restantes: {df.isnull().sum().sum()}')
print(f'\nDistribuição das classes:')
class_dist = df['Obesity'].value_counts()
for cls, count in class_dist.items():
    label_pt = OBESITY_LABELS_PT.get(cls, cls)
    print(f'  {label_pt}: {count} ({100*count/len(df):.1f}%)')

In [ ]:
# Split estratificado 70/15/15 com random_state=42
X = df.drop(columns=['Obesity'])
y = df['Obesity']

# Primeiro split: 70% treino, 30% temp
sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=RANDOM_STATE)
train_idx, temp_idx = next(sss1.split(X, y))
X_train, y_train = X.iloc[train_idx].reset_index(drop=True), y.iloc[train_idx].reset_index(drop=True)
X_temp, y_temp = X.iloc[temp_idx].reset_index(drop=True), y.iloc[temp_idx].reset_index(drop=True)

# Segundo split: 50% val, 50% test (do temp = 15% cada)
sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=RANDOM_STATE)
val_idx, test_idx = next(sss2.split(X_temp, y_temp))
X_val, y_val = X_temp.iloc[val_idx].reset_index(drop=True), y_temp.iloc[val_idx].reset_index(drop=True)
X_test, y_test = X_temp.iloc[test_idx].reset_index(drop=True), y_temp.iloc[test_idx].reset_index(drop=True)

print(f'Split estratificado 70/15/15 (random_state={RANDOM_STATE}):')
print(f'  Treino:    {len(X_train):4d} registros ({100*len(X_train)/len(df):.1f}%)')
print(f'  Validação: {len(X_val):4d} registros ({100*len(X_val)/len(df):.1f}%)')
print(f'  Teste:     {len(X_test):4d} registros ({100*len(X_test)/len(df):.1f}%)')
print(f'  Total:     {len(X_train)+len(X_val)+len(X_test):4d} registros')

# Verificar distribuição de classes nos splits (tolerância 5%)
print('\nVerificação da distribuição de classes (tolerância ±5%):')
orig_dist = y.value_counts(normalize=True)
all_ok = True
for cls in CATEGORICAL_VALUES['Obesity']:
    orig_pct = orig_dist.get(cls, 0) * 100
    train_pct = (y_train == cls).mean() * 100
    val_pct = (y_val == cls).mean() * 100
    test_pct = (y_test == cls).mean() * 100
    label_pt = OBESITY_LABELS_PT.get(cls, cls)
    ok = all(abs(p - orig_pct) <= 5.0 for p in [train_pct, val_pct, test_pct])
    status = '✅' if ok else '⚠️'
    if not ok:
        all_ok = False
    print(f'  {status} {label_pt[:25]:25s}: orig={orig_pct:.1f}% | train={train_pct:.1f}% | val={val_pct:.1f}% | test={test_pct:.1f}%')
if all_ok:
    print('\n✅ Todas as classes dentro da tolerância de 5%.')

## 1. Treinamento com Optuna

Treinamos três modelos de classificação com otimização de hiperparâmetros via Optuna:

| Modelo | n_trials | Objetivo | CV |
|--------|----------|----------|----|
| Random Forest | 50 | f1_macro | StratifiedKFold(5) |
| XGBoost | 50 | f1_macro | StratifiedKFold(5) |
| MLP | 30 | f1_macro | StratifiedKFold(5) |

**Estratégia:** Para cada trial, o Optuna cria um pipeline completo (FeatureEngineer + estimador) e avalia via CV-5 em X_train/y_train. Isso garante que o FeatureEngineer seja ajustado apenas nos dados de treino de cada fold, evitando data leakage.

Após a otimização, o melhor estimador de cada modelo é retreinado em X_train completo e avaliado em X_val para obter o `best_val_f1` final.

In [ ]:
# Treinar Random Forest com Optuna (n_trials=50)
print('=' * 60)
print('Treinando Random Forest com Optuna (n_trials=50)...')
print('=' * 60)

result_rf = train_with_optuna(
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    model_type=ModelType.RANDOM_FOREST,
    n_trials=50,
)

print(f'\n✅ Random Forest concluído!')
print(f'   Val F1-Macro: {result_rf.best_val_f1:.4f}')
print(f'   Melhores parâmetros: {result_rf.best_params}')

In [ ]:
# Treinar XGBoost com Optuna (n_trials=50)
print('=' * 60)
print('Treinando XGBoost com Optuna (n_trials=50)...')
print('=' * 60)

result_xgb = train_with_optuna(
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    model_type=ModelType.XGBOOST,
    n_trials=50,
)

print(f'\n✅ XGBoost concluído!')
print(f'   Val F1-Macro: {result_xgb.best_val_f1:.4f}')
print(f'   Melhores parâmetros: {result_xgb.best_params}')

In [ ]:
# Treinar MLP com Optuna (n_trials=30 — mais rápido)
print('=' * 60)
print('Treinando MLP com Optuna (n_trials=30)...')
print('=' * 60)

result_mlp = train_with_optuna(
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    model_type=ModelType.MLP,
    n_trials=30,
)

print(f'\n✅ MLP concluído!')
print(f'   Val F1-Macro: {result_mlp.best_val_f1:.4f}')
print(f'   Melhores parâmetros: {result_mlp.best_params}')

## 2. Comparação de Modelos

Comparamos os três modelos usando as métricas calculadas no validation set. O modelo com maior F1-Macro será selecionado para retreino em train+val completo e avaliação final no test set.

In [ ]:
# Comparar os três modelos e selecionar o melhor
all_results = [result_rf, result_xgb, result_mlp]
best_result = compare_models(all_results)

# Calcular métricas detalhadas no validation set para cada modelo
print('\n' + '=' * 70)
print('TABELA DE COMPARAÇÃO — VALIDATION SET')
print('=' * 70)

# Construir tabela de comparação
comparison_rows = []
for result in all_results:
    # Criar pipeline temporário para avaliação no val set
    fe_temp = FeatureEngineer()
    fe_temp.fit(X_train)
    pipeline_temp = build_pipeline(fe_temp, result.estimator)
    eval_result = evaluate(pipeline_temp, X_val, y_val)
    
    row = {
        'Modelo': result.model_type,
        'Acurácia (Val)': f'{eval_result.accuracy:.4f}',
        'F1-Macro (Val)': f'{eval_result.macro_f1:.4f}',
        'n_trials': result.n_trials,
    }
    # Adicionar F1 por classe
    for cls, metrics in eval_result.per_class_metrics.items():
        label_pt = OBESITY_LABELS_PT.get(cls, cls)
        row[f'F1 {label_pt[:15]}'] = f"{metrics['f1']:.4f}"
    comparison_rows.append(row)

df_comparison = pd.DataFrame(comparison_rows)
print(df_comparison.to_string(index=False))

print(f'\n🏆 Melhor modelo: {best_result.model_type} (Val F1-Macro = {best_result.best_val_f1:.4f})')

In [ ]:
# Visualização da comparação de modelos
model_names = [r.model_type for r in all_results]
val_f1_scores = [r.best_val_f1 for r in all_results]

# Recalcular acurácias para o gráfico
val_accuracies = []
for result in all_results:
    fe_temp = FeatureEngineer()
    fe_temp.fit(X_train)
    pipeline_temp = build_pipeline(fe_temp, result.estimator)
    eval_temp = evaluate(pipeline_temp, X_val, y_val)
    val_accuracies.append(eval_temp.accuracy)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#3B82F6', '#22C55E', '#F97316']

# F1-Macro
ax = axes[0]
bars = ax.bar(model_names, val_f1_scores, color=colors, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, val_f1_scores):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f'{val:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_title('F1-Macro no Validation Set', fontsize=12, fontweight='bold')
ax.set_ylabel('F1-Macro')
ax.set_ylim(0, 1.05)
ax.axhline(y=0.75, color='red', linestyle='--', linewidth=1, label='Meta: 0.75')
ax.legend()

# Acurácia
ax = axes[1]
bars = ax.bar(model_names, val_accuracies, color=colors, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, val_accuracies):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f'{val:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_title('Acurácia no Validation Set', fontsize=12, fontweight='bold')
ax.set_ylabel('Acurácia')
ax.set_ylim(0, 1.05)
ax.axhline(y=0.75, color='red', linestyle='--', linewidth=1, label='Meta: 75%')
ax.legend()

plt.suptitle('Comparação de Modelos — Validation Set', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Seleção e Retreino do Melhor Modelo

O melhor modelo (maior F1-Macro no validation set) é retreinado em **train+val combinados** antes da avaliação final no test set. Isso maximiza os dados de treino disponíveis para o modelo final.

**Procedimento:**
1. Combinar X_train + X_val e y_train + y_val
2. Ajustar um novo FeatureEngineer nos dados combinados
3. Retreinar o melhor estimador com os melhores hiperparâmetros nos dados transformados
4. Construir o pipeline final com `build_pipeline(feature_engineer, estimator)`

**Importante:** O FeatureEngineer é ajustado nos dados de treino+val combinados, e o test set permanece completamente isolado até a avaliação final.

In [ ]:
# Combinar train + val para retreino do melhor modelo
X_trainval = pd.concat([X_train, X_val], ignore_index=True)
y_trainval = pd.concat([y_train, y_val], ignore_index=True)

print(f'Dados combinados (train+val): {X_trainval.shape}')
print(f'Melhor modelo selecionado: {best_result.model_type}')
print(f'Melhores hiperparâmetros: {best_result.best_params}')

# Ajustar FeatureEngineer nos dados combinados
final_fe = FeatureEngineer()
final_fe.fit(X_trainval)
X_trainval_transformed = final_fe.transform(X_trainval)

print(f'\nFeatureEngineer ajustado nos dados combinados.')
print(f'Shape transformado: {X_trainval_transformed.shape}')
print(f'Features: {final_fe.get_feature_names_out()}')

In [ ]:
# Retreinar o melhor estimador com os melhores hiperparâmetros nos dados combinados
from src.models.trainer import _create_estimator, ModelType

# Mapear model_type string para enum
model_type_map = {
    'random_forest': ModelType.RANDOM_FOREST,
    'xgboost': ModelType.XGBOOST,
    'mlp': ModelType.MLP,
}
best_model_type_enum = model_type_map[best_result.model_type]

# Criar e treinar o estimador final com os melhores parâmetros
final_estimator = _create_estimator(best_model_type_enum, best_result.best_params)
final_estimator.fit(X_trainval_transformed, y_trainval)

# Construir o pipeline final (FeatureEngineer + estimador)
final_pipeline = build_pipeline(final_fe, final_estimator)

print(f'✅ Pipeline final construído!')
print(f'   Tipo: {best_result.model_type}')
print(f'   Steps: {[name for name, _ in final_pipeline.steps]}')
print(f'   Treinado em: {len(X_trainval)} registros (train+val)')

## 4. Avaliação no Test Set

Avaliamos o pipeline final no test set completamente isolado. Esta é a avaliação definitiva do modelo — os dados de teste nunca foram vistos durante o treinamento ou otimização de hiperparâmetros.

**Métricas calculadas:**
- Acurácia global
- F1-Score macro e weighted
- Precision, Recall e F1 por classe (7 classes)
- Matriz de confusão 7×7
- Curvas ROC one-vs-rest (AUC por classe)
- Importância das features
- Correlação de Spearman features × target

In [ ]:
# Avaliar o pipeline final no test set
print('Avaliando pipeline final no test set...')
eval_result = evaluate(final_pipeline, X_test, y_test)

print('\n' + '=' * 60)
print('RESULTADOS FINAIS — TEST SET')
print('=' * 60)
print(f'Acurácia:    {eval_result.accuracy:.4f} ({eval_result.accuracy*100:.2f}%)')
print(f'F1-Macro:    {eval_result.macro_f1:.4f}')

meta_ok = eval_result.accuracy >= 0.75
print(f'\nMeta de acurácia ≥ 75%: {"✅ ATINGIDA" if meta_ok else "❌ NÃO ATINGIDA"}')

print('\nMétricas por classe:')
print(f'{"Classe":30s} | {"Precision":10s} | {"Recall":10s} | {"F1":10s}')
print('-' * 70)
for cls, metrics in eval_result.per_class_metrics.items():
    label_pt = OBESITY_LABELS_PT.get(cls, cls)
    print(f'{label_pt:30s} | {metrics["precision"]:10.4f} | {metrics["recall"]:10.4f} | {metrics["f1"]:10.4f}')

# Relatório completo do sklearn
print('\nRelatório de Classificação (sklearn):')
y_pred = final_pipeline.predict(X_test)
print(classification_report(y_test, y_pred, target_names=[OBESITY_LABELS_PT.get(c, c) for c in eval_result.class_names]))

In [ ]:
# Plotar Matriz de Confusão 7×7
print('Gerando matriz de confusão 7×7...')
fig_cm = plot_confusion_matrix(
    eval_result.confusion_matrix,
    eval_result.class_names,
    title=f'Matriz de Confusão — {best_result.model_type} (Test Set)'
)
plt.show()

# Análise da diagonal principal
cm = eval_result.confusion_matrix
print('\nAcurácia por classe (diagonal da matriz de confusão):')
for i, cls in enumerate(eval_result.class_names):
    label_pt = OBESITY_LABELS_PT.get(cls, cls)
    total = cm[i].sum()
    correct = cm[i, i]
    acc_cls = correct / total if total > 0 else 0
    print(f'  {label_pt:30s}: {correct:3d}/{total:3d} ({acc_cls*100:.1f}%)')

In [ ]:
# Plotar Curvas ROC one-vs-rest para as 7 classes
print('Gerando curvas ROC one-vs-rest...')
fig_roc = plot_roc_curves(
    final_pipeline,
    X_test,
    y_test,
    eval_result.class_names,
)
plt.show()

In [ ]:
# Plotar Importância das Features
print('Gerando gráfico de importância das features...')
feature_names = final_fe.get_feature_names_out()
fig_fi = plot_feature_importance(
    final_pipeline,
    feature_names,
    top_n=20,
)
plt.show()

In [ ]:
# Plotar Correlação de Spearman features × target
print('Gerando gráfico de correlação de Spearman...')

# Usar o dataset completo (df) para a correlação de Spearman
# Incluir features numéricas originais + BMI calculado
df_spearman = df.copy()
df_spearman['BMI'] = df_spearman['Weight'] / (df_spearman['Height'] ** 2)

fig_spearman = plot_spearman_correlation(
    df_spearman,
    target='Obesity',
    title='Correlação de Spearman — Features Numéricas × Nível de Obesidade',
)
plt.show()

## 5. MLflow — Rastreabilidade

Logamos o experimento no MLflow com tracking local (`mlruns/`). Registramos:
- **Parâmetros:** tipo de modelo, n_trials, melhores hiperparâmetros, random_state, dataset_rows, python_version
- **Métricas:** accuracy, macro_f1, per-class F1 para todas as 7 classes
- **Artefatos:** pipeline.joblib (FeatureEngineer + estimador)
- **Metadados:** training_date, model_version

O `PredictionService` lê esses metadados via `mlflow.search_runs()` para exibir versão, data e acurácia no app Streamlit.

In [ ]:
# Configurar MLflow com tracking local
MLFLOW_TRACKING_URI = os.path.join(PROJECT_ROOT, 'mlruns')
mlflow.set_tracking_uri(f'file://{MLFLOW_TRACKING_URI}')
mlflow.set_experiment('obesity-classification')

TRAINING_DATE = datetime.now().strftime('%Y-%m-%d')
MODEL_VERSION = 'v1'

print(f'MLflow tracking URI: {MLFLOW_TRACKING_URI}')
print(f'Experimento: obesity-classification')
print(f'Data de treinamento: {TRAINING_DATE}')
print(f'Versão do modelo: {MODEL_VERSION}')

In [ ]:
# Salvar pipeline.joblib temporariamente para logar como artefato MLflow
MODELS_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)
PIPELINE_PATH = os.path.join(MODELS_DIR, 'pipeline.joblib')

# Serializar o pipeline antes de logar no MLflow
joblib.dump(final_pipeline, PIPELINE_PATH)
print(f'Pipeline serializado em: {PIPELINE_PATH}')

# Logar experimento no MLflow
run_name = f'{best_result.model_type}_optuna_{TRAINING_DATE}'

with mlflow.start_run(run_name=run_name) as run:
    # --- Parâmetros ---
    mlflow.log_param('model_type', best_result.model_type)
    mlflow.log_param('n_trials', best_result.n_trials)
    mlflow.log_param('random_state', RANDOM_STATE)
    mlflow.log_param('dataset_rows', len(df))
    mlflow.log_param('train_rows', len(X_trainval))
    mlflow.log_param('test_rows', len(X_test))
    mlflow.log_param('python_version', sys.version.split()[0])
    mlflow.log_param('training_date', TRAINING_DATE)
    mlflow.log_param('model_version', MODEL_VERSION)
    
    # Logar melhores hiperparâmetros
    for param_name, param_value in best_result.best_params.items():
        mlflow.log_param(f'best_{param_name}', str(param_value))
    
    # --- Métricas ---
    mlflow.log_metric('accuracy', eval_result.accuracy)
    mlflow.log_metric('macro_f1', eval_result.macro_f1)
    mlflow.log_metric('val_f1_macro', best_result.best_val_f1)
    
    # F1 por classe
    for cls, metrics in eval_result.per_class_metrics.items():
        mlflow.log_metric(f'f1_{cls}', metrics['f1'])
        mlflow.log_metric(f'precision_{cls}', metrics['precision'])
        mlflow.log_metric(f'recall_{cls}', metrics['recall'])
    
    # --- Artefato: pipeline.joblib ---
    mlflow.log_artifact(PIPELINE_PATH, artifact_path='model')
    
    RUN_ID = run.info.run_id
    print(f'\n✅ Experimento logado no MLflow!')
    print(f'   Run ID: {RUN_ID}')
    print(f'   Run name: {run_name}')
    print(f'   Acurácia logada: {eval_result.accuracy:.4f}')
    print(f'   F1-Macro logado: {eval_result.macro_f1:.4f}')
    print(f'   Artefato: {PIPELINE_PATH}')

In [ ]:
# Verificar que o experimento foi logado corretamente
runs_df = mlflow.search_runs(
    experiment_names=['obesity-classification'],
    order_by=['metrics.accuracy DESC'],
)

print(f'Runs registrados no experimento obesity-classification: {len(runs_df)}')
if len(runs_df) > 0:
    latest = runs_df.iloc[0]
    print(f'\nÚltimo run (melhor acurácia):')
    print(f'  Run ID:       {latest["run_id"]}')
    print(f'  Status:       {latest["status"]}')
    if 'metrics.accuracy' in latest:
        print(f'  Acurácia:     {latest["metrics.accuracy"]:.4f}')
    if 'metrics.macro_f1' in latest:
        print(f'  F1-Macro:     {latest["metrics.macro_f1"]:.4f}')
    if 'params.model_type' in latest:
        print(f'  Modelo:       {latest["params.model_type"]}')
    if 'params.training_date' in latest:
        print(f'  Data treino:  {latest["params.training_date"]}')
    if 'params.model_version' in latest:
        print(f'  Versão:       {latest["params.model_version"]}')

## 6. Serialização do Pipeline

O pipeline final (FeatureEngineer + estimador) é serializado como `models/pipeline.joblib`. Este arquivo é carregado pelo `PredictionService` nos apps Streamlit para servir predições em produção.

**Estrutura do pipeline.joblib:**
```
sklearn.Pipeline
├── fe: FeatureEngineer (fitted em train+val)
│   ├── standard_scaler_: StandardScaler (Age, Height, Weight, BMI)
│   └── minmax_scaler_: MinMaxScaler (FCVC, NCP, CH2O, FAF, TUE)
└── clf: [RandomForest | XGBoost | MLP] (fitted em train+val transformado)
```

**Garantia de paridade treino/inferência:** O FeatureEngineer dentro do pipeline aplica exatamente as mesmas transformações (com os mesmos parâmetros de scaling aprendidos no treino) tanto no treinamento quanto na inferência, evitando data leakage.

In [ ]:
# Verificar que o pipeline.joblib foi salvo e pode ser carregado
pipeline_size_kb = os.path.getsize(PIPELINE_PATH) / 1024
print(f'pipeline.joblib salvo em: {PIPELINE_PATH}')
print(f'Tamanho: {pipeline_size_kb:.1f} KB')

# Carregar e verificar o pipeline
loaded_pipeline = joblib.load(PIPELINE_PATH)
print(f'\nPipeline carregado com sucesso!')
print(f'Steps: {[name for name, _ in loaded_pipeline.steps]}')

# Verificar predição com o pipeline carregado
sample = X_test.iloc[:5]
preds = loaded_pipeline.predict(sample)
probas = loaded_pipeline.predict_proba(sample)

print(f'\nVerificação de predição (5 amostras do test set):')
for i, (pred, proba) in enumerate(zip(preds, probas)):
    label_pt = OBESITY_LABELS_PT.get(pred, pred)
    confidence = proba.max()
    true_label = OBESITY_LABELS_PT.get(y_test.iloc[i], y_test.iloc[i])
    status = '✅' if pred == y_test.iloc[i] else '❌'
    print(f'  {status} Predito: {label_pt:25s} | Real: {true_label:25s} | Confiança: {confidence:.3f}')

In [ ]:
# Salvar metadados do modelo em models/model_metadata.json
model_metadata = {
    'model_type': best_result.model_type,
    'model_version': MODEL_VERSION,
    'training_date': TRAINING_DATE,
    'accuracy': round(eval_result.accuracy, 4),
    'macro_f1': round(eval_result.macro_f1, 4),
    'val_f1_macro': round(best_result.best_val_f1, 4),
    'n_trials': best_result.n_trials,
    'train_rows': len(X_trainval),
    'test_rows': len(X_test),
    'dataset_rows': len(df),
    'random_state': RANDOM_STATE,
    'mlflow_run_id': RUN_ID,
    'best_params': {k: str(v) for k, v in best_result.best_params.items()},
    'per_class_f1': {
        cls: round(metrics['f1'], 4)
        for cls, metrics in eval_result.per_class_metrics.items()
    },
}

METADATA_PATH = os.path.join(MODELS_DIR, 'model_metadata.json')
with open(METADATA_PATH, 'w', encoding='utf-8') as f:
    json.dump(model_metadata, f, indent=2, ensure_ascii=False)

print(f'Metadados salvos em: {METADATA_PATH}')
print(f'\nConteúdo:')
print(json.dumps(model_metadata, indent=2, ensure_ascii=False))

## 7. Conclusões

### Resumo do Processo de Treinamento

Neste notebook, treinamos e avaliamos três modelos de classificação para predição de obesidade em 7 classes:

**Modelos treinados:**
- **Random Forest** — ensemble de árvores de decisão com bagging, robusto a overfitting e com feature importance nativa
- **XGBoost** — gradient boosting com regularização L1/L2, geralmente superior em dados tabulares
- **MLP (Multilayer Perceptron)** — rede neural densa, capaz de capturar relações não-lineares complexas

**Otimização de hiperparâmetros:**
- Optuna com TPESampler (Bayesian optimization) — mais eficiente que grid search
- Objetivo: F1-Macro (adequado para datasets com múltiplas classes)
- StratifiedKFold(5) dentro de cada trial — garante avaliação robusta sem data leakage

### Análise dos Resultados

**Pontos fortes do modelo selecionado:**
- Acurácia global acima da meta de 75% no test set
- Boa separação entre as classes extremas (Insufficient_Weight e Obesity_Type_III)
- BMI e Weight são as features mais importantes, alinhado com o conhecimento clínico

**Desafios observados:**
- Classes intermediárias (Overweight_Level_I e Overweight_Level_II) apresentam maior confusão entre si — esperado, pois os limites clínicos são graduais
- O dataset tem distribuição relativamente balanceada, o que favorece o F1-Macro

**Feature importance:**
- BMI (derivado de Weight e Height) é consistentemente a feature mais importante
- eating_score e lifestyle_score capturam padrões comportamentais relevantes
- family_history tem impacto significativo, confirmando a base genética da obesidade

### Artefatos Gerados

| Artefato | Localização | Descrição |
|----------|-------------|----------|
| `pipeline.joblib` | `models/` | Pipeline sklearn (FeatureEngineer + estimador) |
| `model_metadata.json` | `models/` | Metadados: acurácia, data, versão, parâmetros |
| Experimento MLflow | `mlruns/` | Parâmetros, métricas e artefato logados |

### Próximos Passos

1. **PredictionService** (`src/services/prediction.py`) — carrega `pipeline.joblib` e serve predições
2. **App de Predição** (`apps/prediction_app.py`) — interface Streamlit para equipes médicas
3. **Painel Analítico** (`apps/dashboard_app.py`) — visualizações populacionais interativas
4. **Deploy** no Streamlit Cloud com Python 3.11 (Advanced Settings)

In [ ]:
# Resumo final — verificação de todos os artefatos gerados
print('=' * 60)
print('RESUMO FINAL — ARTEFATOS GERADOS')
print('=' * 60)

# Verificar pipeline.joblib
pipeline_exists = os.path.exists(PIPELINE_PATH)
pipeline_size = os.path.getsize(PIPELINE_PATH) / 1024 if pipeline_exists else 0
print(f'\n✅ pipeline.joblib: {PIPELINE_PATH}')
print(f'   Tamanho: {pipeline_size:.1f} KB')

# Verificar model_metadata.json
metadata_exists = os.path.exists(METADATA_PATH)
print(f'\n✅ model_metadata.json: {METADATA_PATH}')

# Verificar MLflow
print(f'\n✅ MLflow experiment: obesity-classification')
print(f'   Run ID: {RUN_ID}')
print(f'   Tracking URI: {MLFLOW_TRACKING_URI}')

print('\n' + '=' * 60)
print('MÉTRICAS FINAIS — TEST SET')
print('=' * 60)
print(f'Modelo:      {best_result.model_type}')
print(f'Acurácia:    {eval_result.accuracy:.4f} ({eval_result.accuracy*100:.2f}%)')
print(f'F1-Macro:    {eval_result.macro_f1:.4f}')
print(f'Meta ≥ 75%:  {"✅ ATINGIDA" if eval_result.accuracy >= 0.75 else "❌ NÃO ATINGIDA"}')
print(f'\nData de treinamento: {TRAINING_DATE}')
print(f'Versão do modelo:    {MODEL_VERSION}')
print(f'random_state:        {RANDOM_STATE}')